# 04 第 1 節｜OpenCV 影像處理與 Tesseract OCR

**適合高中生的版本：4 組必做練習＋Optional。** 每一組先看 Markdown 程式範例，再於下方空白 Cell 照寫。較難的程式已放到 Optional，可以直接執行。

本教材沿用老師提供的 4 張車牌圖片 `04_素材/plate_01.png`～`plate_04.png`；**它們已是裁切好的車牌影像**，所以先練習影像讀取、灰階與 OCR，**不把「從整台車找出車牌」誤當成已完成**。

**開始前：**將教學 ZIP 解壓縮到 Colab `/content/`。Notebook 跟 `04_素材/` 必須位於同一工作資料夾。若在 VS Code 接 Colab 伺服器，也請先把素材上傳到遠端。


## 0. 安裝（直接執行，不需要學生照打）


In [ ]:
# Install the OCR program in Colab.
# 在 Colab 安裝 Tesseract OCR 系統程式。
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr > /dev/null

# Install the Python OCR library.
# 安裝 Python 使用 OCR 的套件。
%pip -q install pytesseract


## 0. 匯入套件（直接執行，不需要學生照打）


In [ ]:
# Import OpenCV for image processing.
# 匯入 OpenCV 進行影像處理。
import cv2

# Import the OCR tool.
# 匯入 Tesseract OCR 介面。
import pytesseract

# Import Matplotlib for displaying image results.
# 匯入 Matplotlib 顯示影像結果。
import matplotlib.pyplot as plt

# Import image display tools.
# 匯入 Notebook 圖片顯示工具。
from PIL import Image
from IPython.display import display

# Prevent minus-sign rendering problems.
# 避免座標軸的負號顯示成方框。
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False


## 練習 1｜讀取一張圖片

先改 `IMAGE_PATH`，觀察不同的車牌照片。

```python
# Choose an image file.
# 指定要讀取的圖片。
IMAGE_PATH = '04_素材/plate_04.png'

# Read the image.
# 讀取圖片。
img = cv2.imread(IMAGE_PATH)

# Display the image with correct colors.
# 用正確的色彩顯示圖片。
display(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))

# Check its shape.
# 查看圖片的高度、寬度與通道數。
print('圖片尺寸：', img.shape)
```


In [ ]:
# 請在此處根據範例練習程式碼


## 練習 2｜灰階與二值化

灰階只有亮度；二值化使用門檻將像素變成黑白。學生可以修改 `THRESHOLD`。

```python
# Convert the image to grayscale.
# 將圖片轉成灰階。
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Set the brightness threshold.
# 設定黑白門檻。
THRESHOLD = 130

# Convert gray pixels to black and white.
# 將灰階影像轉成黑白影像。
_, binary = cv2.threshold(gray, THRESHOLD, 255, cv2.THRESH_BINARY)

# Show both results.
# 比較灰階與二值化結果。
display(Image.fromarray(gray))
display(Image.fromarray(binary))
```


In [ ]:
# 請在此處根據範例練習程式碼


## 練習 3｜模糊與邊緣偵測

先使用 Gaussian Blur 降低雜訊，再用 Canny 找影像中的邊緣。

```python
# Blur the grayscale image.
# 模糊影像，減少雜訊。
blur = cv2.GaussianBlur(gray, (5, 5), 0)

# Detect image edges.
# 使用 Canny 偵測邊緣。
edges = cv2.Canny(blur, 80, 160)

# Show the edge image.
# 顯示邊緣偵測結果。
display(Image.fromarray(edges))
```


In [ ]:
# 請在此處根據範例練習程式碼


## 練習 4｜使用 Tesseract 辨識車牌文字

照片已經裁切好車牌，所以讓 OCR 直接讀取灰階圖片。**辨識結果可能有錯誤，不代表程式失敗。**

```python
# Recognize a single line of text.
# 將車牌視為一行文字進行辨識。
text = pytesseract.image_to_string(gray, config='--psm 7')

# Print recognized text.
# 印出辨識到的文字。
print('OCR 辨識結果：', text.strip())
```


In [ ]:
# 請在此處根據範例練習程式碼


## Optional｜直接執行，觀察影像處理技巧


### Optional｜縮放圖片


In [ ]:
# Resize to half the original image size.
# 將圖片縮小為原本的一半。
small = cv2.resize(img, None, fx=0.5, fy=0.5)

# Display the smaller picture.
# 顯示縮小後的影像。
display(Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB)))


### Optional｜亮度直方圖


In [ ]:
# Draw the grayscale intensity distribution.
# 繪製灰階亮度分布。
plt.hist(gray.ravel(), bins=32, range=(0, 256))

# Label the axes.
# 標示圖表座標軸。
plt.xlabel('Brightness')
plt.ylabel('Pixel count')
plt.title('Grayscale histogram')
plt.show()


### Optional｜使用 4 張車牌比較 OCR（由程式自動逐張辨識）


In [ ]:
# Compare OCR results for all four supplied plate images.
# 讓程式逐張比較四張車牌的 OCR 結果。
for n in range(1, 5):
    # Read each image in grayscale.
    # 用灰階讀取每一張圖片。
    photo = cv2.imread(f'04_素材/plate_0{n}.png', cv2.IMREAD_GRAYSCALE)

    # Read the characters.
    # 辨識車牌文字。
    answer = pytesseract.image_to_string(photo, config='--psm 7')

    # Show the file name and OCR result.
    # 顯示圖片名稱與文字結果。
    print(f'plate_0{n}.png：', answer.strip())


**小結：**OpenCV 負責圖片前處理；Tesseract 負責辨識文字。這份素材已經完成車牌裁切，因此沒有示範如何在整張車輛照片中定位車牌。將來如果要自動尋找整張照片裡的車牌，還需要額外的偵測步驟。
